## Tarea 3
Programa Experto en Inteligencia Artificial con Python  
ML3009 - Series de Tiempo  
Estudiante: Natalia Bonilla Villalobos.  
**Métodos de suavizado exponencial**

<a id="menu"></a>

# Menú
- [Ejercicio 1](#ej1)
- [Ejercicio 2](#ej2)
- [Ejercicio 3](#ej3)

In [83]:
import math
import warnings
import statistics
import numpy as np
import pandas as pd
from scipy import stats, signal
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from scipy.stats import shapiro
from statsmodels.tsa.api import ExponentialSmoothing  # Holt-Winters
from statsmodels.tsa.seasonal import seasonal_decompose

from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.simplefilter("ignore", ConvergenceWarning)
from abc import ABCMeta, abstractmethod

warnings.filterwarnings("ignore")
from numpy import corrcoef

from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense

import plotly.io as pio
pio.renderers.default = "notebook_connected"

## Modelos Predictivos

In [2]:
class BasePrediccion(metaclass=ABCMeta):
    @abstractmethod
    def forecast(self):
        pass


class Prediccion(BasePrediccion):
    def __init__(self, modelo):
        self.__modelo = modelo

    @property
    def modelo(self):
        return self.__modelo

    @modelo.setter
    def modelo(self, modelo):
        if isinstance(modelo, Modelo):
            self.__modelo = modelo
        else:
            warnings.warn("El objeto debe ser una instancia de Modelo.")


class meanfPrediccion(Prediccion):
    def __init__(self, modelo):
        super().__init__(modelo)

    def forecast(self, steps=1):
        res = []
        for i in range(steps):
            res.append(self.modelo.coef)

        start = self.modelo.ts.index[-1]
        freq = self.modelo.ts.index.freqstr
        fechas = pd.date_range(start=start, periods=steps + 1, freq=freq)
        fechas = fechas.delete(0)
        res = pd.Series(res, index=fechas)
        return res


class naivePrediccion(Prediccion):
    def __init__(self, modelo):
        super().__init__(modelo)

    def forecast(self, steps=1):
        res = []
        for i in range(steps):
            res.append(self.modelo.coef)

        start = self.modelo.ts.index[-1]
        freq = self.modelo.ts.index.freqstr
        fechas = pd.date_range(start=start, periods=steps + 1, freq=freq)
        fechas = fechas.delete(0)
        res = pd.Series(res, index=fechas)
        return res


class snaivePrediccion(Prediccion):
    def __init__(self, modelo):
        super().__init__(modelo)

    def forecast(self, steps=1):
        res = []
        pos = 0
        for i in range(steps):
            if pos >= len(self.modelo.coef):
                pos = 0
            res.append(self.modelo.coef[pos])
            pos = pos + 1

        start = self.modelo.ts.index[-1]
        freq = self.modelo.ts.index.freqstr
        fechas = pd.date_range(start=start, periods=steps + 1, freq=freq)
        fechas = fechas.delete(0)
        res = pd.Series(res, index=fechas)
        return res


class driftPrediccion(Prediccion):
    def __init__(self, modelo):
        super().__init__(modelo)

    def forecast(self, steps=1):
        res = []
        for i in range(steps):
            res.append(self.modelo.ts[-1] + self.modelo.coef * i)

        start = self.modelo.ts.index[-1]
        freq = self.modelo.ts.index.freqstr
        fechas = pd.date_range(start=start, periods=steps + 1, freq=freq)
        fechas = fechas.delete(0)
        res = pd.Series(res, index=fechas)
        return res


class BaseModelo(metaclass=ABCMeta):
    @abstractmethod
    def fit(self):
        pass


class Modelo(BaseModelo):
    def __init__(self, ts):
        self.__ts = ts
        self._coef = None

    @property
    def ts(self):
        return self.__ts

    @ts.setter
    def ts(self, ts):
        if isinstance(ts, pd.core.series.Series):
            if ts.index.freqstr != None:
                self.__ts = ts
            else:
                warnings.warn(
                    "ERROR: No se indica la frecuencia de la serie de tiempo."
                )
        else:
            warnings.warn(
                "ERROR: El parámetro ts no es una instancia de serie de tiempo."
            )

    @property
    def coef(self):
        return self._coef


class meanf(Modelo):
    def __init__(self, ts):
        super().__init__(ts)

    def fit(self):
        self._coef = statistics.mean(self.ts)
        res = meanfPrediccion(self)
        return res


class naive(Modelo):
    def __init__(self, ts):
        super().__init__(ts)

    def fit(self):
        self._coef = self.ts[-1]
        res = naivePrediccion(self)
        return res


class snaive(Modelo):
    def __init__(self, ts):
        super().__init__(ts)

    def fit(self, h=1):
        self._coef = self.ts.values[-h:]
        res = snaivePrediccion(self)
        return res


class drift(Modelo):
    def __init__(self, ts):
        super().__init__(ts)

    def fit(self):
        self._coef = (self.ts[-1] - self.ts[0]) / len(self.ts)
        res = driftPrediccion(self)
        return res

## Cálculo de errores

In [3]:
class ts_error:
    def __init__(self, preds, real, nombres=None):
        self.__preds = preds
        self.__real = real
        self.__nombres = nombres

    @property
    def preds(self):
        return self.__preds

    @preds.setter
    def preds(self, preds):
        if isinstance(preds, pd.core.series.Series) or isinstance(preds, numpy.ndarray):
            self.__preds = [preds]
        elif isinstance(preds, list):
            self.__preds = preds
        else:
            warnings.warn(
                "ERROR: El parámetro preds debe ser una serie de tiempo o una lista de series de tiempo."
            )

    @property
    def real(self):
        return self.__real

    @real.setter
    def real(self, real):
        self.__real = real

    @property
    def nombres(self):
        return self.__nombres

    @nombres.setter
    def nombres(self, nombres):
        if isinstance(nombres, str):
            nombres = [nombres]
        if len(nombres) == len(self.__preds):
            self.__nombres = nombres
        else:
            warnings.warn("ERROR: Los nombres no calzan con la cantidad de métodos.")

    def RSS(self):
        res = []
        for pred in self.preds:
            res.append(sum((pred - self.real) ** 2))
        return res

    def MSE(self):
        return [pred / len(self.real) for pred in self.RSS()]

    def RMSE(self):
        return [math.sqrt(pred) for pred in self.MSE()]

    def RE(self):
        res = []
        for pred in self.preds:
            res.append(sum(abs(self.real - pred)) / sum(abs(self.real)))
        return res

    def CORR(self):
        res = []
        for pred in self.preds:
            corr = corrcoef(self.real, pred)[0, 1]
            res.append(0 if math.isnan(corr) else corr)
        return res

    def df_errores(self):
        res = pd.DataFrame(
            {
                "MSE": self.MSE(),
                "RMSE": self.RMSE(),
                "RE": self.RE(),
                "CORR": self.CORR(),
            }
        )
        if self.nombres is not None:
            res.index = self.nombres
        return res

    def __escalar(self):
        res = self.df_errores()
        for nombre in res.columns.values:
            res[nombre] = res[nombre] - min(res[nombre])
            res[nombre] = res[nombre] / max(res[nombre]) * 100
        return res

    def plot_errores(self):
        plt.figure(figsize=(8, 8))
        df = self.__escalar()
        if len(df) == 1:
            df.loc[0] = 100

        N = len(df.columns.values)
        angles = [n / float(N) * 2 * math.pi for n in range(N)]
        angles += angles[:1]

        ax = plt.subplot(111, polar=True)

        ax.set_theta_offset(math.pi / 2)
        ax.set_theta_direction(-1)

        plt.xticks(angles[:-1], df.columns.values)

        ax.set_rlabel_position(0)
        plt.yticks(
            [0, 25, 50, 75, 100],
            ["0%", "25%", "50%", "75%", "100%"],
            color="grey",
            size=10,
        )
        plt.ylim(-10, 110)

        for i in df.index.values:
            p = df.loc[i].values.tolist()
            p = p + p[:1]
            ax.plot(angles, p, linewidth=1, linestyle="solid", label=i)
            ax.fill(angles, p, alpha=0.1)

        plt.legend(loc="best")
        plt.show()

    def plotly_errores(self):
        df = self.__escalar()
        etqs = df.columns.values.tolist()
        etqs = etqs + etqs[:1]
        if len(df) == 1:
            df.loc[0] = 100

        fig = go.Figure()

        for i in df.index.values:
            p = df.loc[i].values.tolist()
            p = p + p[:1]
            fig.add_trace(go.Scatterpolar(r=p, theta=etqs, fill="toself", name=i))

        fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[-10, 110])))

        return fig

<a id="ej1"></a>
# Ejercicio 1 
[30 puntos] El archivo `emision_monetaria_CR.csv` contiene la emisión monetaria del BCCR como saldos a fin de mes en millones de colones, desde enero de 1986 hasta marzo del 2026.
Con la tabla realice lo siguiente:
- a) Verifique si hay fechas faltantes y de ser así relice la corrección mediante un suavizado (Utilice el valor que usted considere).
- b) Convierta a serie de tiempo.
- c) Usando 3 meses para pruebas y el resto de fechas para entrenamiento genere modelos utilizando `HOLT-WINTERS`,` HOLT-WINTERS Calibrado `y `Redes Neuronales`, luego en un solo gráfico muestre la serie de entrenamiento, la serie de prueba y el resultado de la predicción de cada uno de los modelos anteriores.
- d) Con un gráfico mida el error de cada uno de los modelos anteriores y determine cual de los modelos es el mejor.
- e) Con el mejor modelo encontrado en el punto anterior genere la predicción de 3 meses, pero esta vez utilizando toda la serie de tiempo. Grafique la serie original y la predicción.

[↑ Volver al Menú](#menu)


### Preparación de los datos

In [85]:
emision_monetaria = pd.read_csv("../../w1/h1/emision_monetaria_CR.csv")
emision_monetaria

,Fecha,Valor
0,1986-01-01,12069.5
1,1986-02-01,12314.0
2,1986-03-01,12920.9
3,1986-04-01,12254.1
4,1986-05-01,12478.1
...,...,...
478,2025-11-01,1595905.1
479,2025-12-01,1625018.7
480,2026-01-01,1491267.8
481,2026-02-01,1471950.0


In [86]:
emision_monetaria.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 483 entries, 0 to 482
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Fecha   483 non-null    object 
 1   Valor   483 non-null    float64
dtypes: float64(1), object(1)
memory usage: 7.7+ KB


In [87]:
emision_monetaria["Valor"] = emision_monetaria["Valor"].astype("float64")
emision_monetaria["Fecha"] = emision_monetaria["Fecha"].astype("datetime64[ns]")
emision_monetaria.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 483 entries, 0 to 482
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Fecha   483 non-null    datetime64[ns]
 1   Valor   483 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 7.7 KB


**Funciones de la Tarea#2**

In [88]:
# función para verificar si hay fechas faltantes en la columna de fecha, con una frecuencia dada
def hay_fechas_faltantes(datos, col_name_date, frequency):
    df = datos.copy()
    start_date = df[col_name_date].min()
    last_date = df[col_name_date].max()
    date_range = pd.date_range(
        start=start_date, end=last_date, freq=frequency
    ).to_list()

    faltantes = pd.Index(date_range).difference(df[col_name_date])
    if len(faltantes) == 0:
        print(f"No, no hay valores faltantes en la columna '{col_name_date}'.")
    else:
        print(
            f"Sí, hay {len(df) - len(date_range)} '{col_name_date}' faltantes."
        )

    return {"date_range": date_range, "faltantes": faltantes}


# función para rellenar las fechas faltantes, derivadas de la función anterior
#
def rellenar_fechas(datos, col_name_date, serie_fechas_completas):
    df = datos.copy()
    # asegurar datetime
    # df[col_name_date] = pd.to_datetime(df[col_name_date])
    df[col_name_date] = df[col_name_date].astype("datetime64[ns]")

    fechas_faltantes = [
        x for x in serie_fechas_completas if x not in df[col_name_date].to_list()
    ]
    print("Primeras 5 filas:\n", fechas_faltantes[:5])

    completando_serie = pd.concat(
        [df, pd.DataFrame({"Fecha": fechas_faltantes})], ignore_index=True
    )  # Se añaden las fechas faltantes al final
    completando_serie.sort_values(
        by=col_name_date, inplace=True
    )  # Se ordenan las fechas de forma ascendente

    print("\nValores faltantes: ", completando_serie[col_name_date].isnull().sum())

    return completando_serie


def to_serie_tiempo(datos, col_name_date, col_name_values, frequency):
    df = datos.copy()
    df.set_index(col_name_date, inplace=True)
    serie_tiempo = df[col_name_values].asfreq(frequency)
    return serie_tiempo


# def suavizado_movil(datos, size):
#     df = datos.copy()
#     suavizado = df.rolling(
#         size, min_periods=1, center=True
#     ).mean()  # min_periods=1 para que no deje NaN al inicio y al final, center=True para que el promedio se centre en la ventana
#     return suavizado


def train_test(serie_tiempo, test_size):
    test = serie_tiempo[-test_size:]  # ultimos para test
    train = serie_tiempo[:-test_size]  # el resto menos el test para train
    print("Train:", len(train), "\nTest :", len(test))

    return {"train": train, "test": test}

# # Imputa valores faltantes utilizando un suavizado móvil, basado en vecinos cercanos (rolling mean).
# # si to_serie es True, se devuelve la serie de tiempo, si no, se devuelve el dataframe con los valores imputados.
def imputar_valores_faltantes_suavizados(datos_rellenados, col_name_values, size, to_serie=False, frequency=None, col_name_date=None):
    df = datos_rellenados.copy()

    suavizado = (
        df[col_name_values].rolling(window=size, center=True, min_periods=1).mean()
    )

    # Primera imputacion
    df[col_name_values] = (df[col_name_values].fillna(suavizado))

    # Verificar si aun quedan NaN
    faltantes = df[col_name_values].isna().sum()

    if faltantes > 0:
        print(f'Aún quedan {faltantes} valores faltantes. Aplicando interpolación...')
        df[col_name_values] = (df[col_name_values].interpolate(method='linear'))
    else:
        print(f'Todos los valores faltantes fueron imputados con suavizado de {size} datos.')

    # convertir a serie temporal
    if to_serie:
        df = to_serie_tiempo(df, col_name_date, col_name_values, frequency)

    return df

### Parte 1
- a) Verifique si hay fechas faltantes y de ser así relice la corrección mediante un suavizado (Utilice el valor que usted considere).

In [89]:
lacking_dates = hay_fechas_faltantes(emision_monetaria, "Fecha", "MS")

No, no hay valores faltantes en la columna 'Fecha'.


### Parte 2

- b) Convierta a serie de tiempo.

In [90]:
emision_monetaria_ts = to_serie_tiempo(
    emision_monetaria, col_name_date="Fecha", col_name_values="Valor", frequency="MS"
)
emision_monetaria_ts

Fecha
1986-01-01      12069.5
1986-02-01      12314.0
1986-03-01      12920.9
1986-04-01      12254.1
1986-05-01      12478.1
                ...    
2025-11-01    1595905.1
2025-12-01    1625018.7
2026-01-01    1491267.8
2026-02-01    1471950.0
2026-03-01    1503491.6
Freq: MS, Name: Valor, Length: 483, dtype: float64

### Parte 3
- c) Usando 3 meses para pruebas y el resto de fechas para entrenamiento genere modelos utilizando `HOLT-WINTERS`,` HOLT-WINTERS Calibrado` y `Redes Neuronales`, luego en un solo gráfico muestre la serie de entrenamiento, la serie de prueba y el resultado de la predicción de cada uno de los modelos anteriores.

In [91]:
train_test_emision_monetaria = train_test(emision_monetaria_ts, 3)
train_em = train_test_emision_monetaria["train"]
test_em = train_test_emision_monetaria["test"]

Train: 480 
Test : 3


#### Holt-Winters

In [92]:
modelo_holt_winters = ExponentialSmoothing(
    train_em, trend="add", seasonal="add", seasonal_periods=12
)
modelo_holt_winters_fit = modelo_holt_winters.fit()
pred_holt_winters = modelo_holt_winters_fit.forecast(3)
pred_holt_winters

2026-01-01    1.489689e+06
2026-02-01    1.449937e+06
2026-03-01    1.467508e+06
Freq: MS, dtype: float64

#### Holt-Winters Calibrado
Para encontrar los mejores parámetros en Holt-Winters, podemos utilizar la fuerza bruta. Para ello, podemos utilizar la siguiente función:

In [93]:
class HW_Prediccion(Prediccion):
    def __init__(self, modelo, alpha, beta, gamma):
        super().__init__(modelo)
        self.__alpha = alpha
        self.__beta = beta
        self.__gamma = gamma

    @property
    def alpha(self):
        return self.__alpha

    @property
    def beta(self):
        return self.__beta

    @property
    def gamma(self):
        return self.__gamma

    def forecast(self, steps=1):
        res = self.modelo.forecast(steps)
        return res


class HW_calibrado(Modelo):
    def __init__(self, ts, test, trend="add", seasonal="add", seasonal_periods=None):
        super().__init__(ts)
        self.__test = test
        self.__modelo = ExponentialSmoothing(
            ts, trend=trend, seasonal=seasonal, seasonal_periods=seasonal_periods
        )

    @property
    def test(self):
        return self.__test

    @test.setter
    def test(self, test):
        if isinstance(test, pd.core.series.Series):
            if test.index.freqstr != None:
                self.__test = test
            else:
                warnings.warn(
                    "ERROR: No se indica la frecuencia de la serie de tiempo."
                )
        else:
            warnings.warn(
                "ERROR: El parámetro ts no es una instancia de serie de tiempo."
            )

    def fit(self, paso=0.1):
        error = float("inf")
        n = np.append(np.arange(0, 1, paso), 1)
        for alpha in n:
            for beta in n:
                for gamma in n:
                    model_fit = self.__modelo.fit(
                        smoothing_level=alpha,
                        smoothing_trend=beta,
                        smoothing_seasonal=gamma,
                    )
                    pred = model_fit.forecast(len(self.test))
                    mse = sum((pred - self.test) ** 2)
                    if mse < error:
                        res_alpha = alpha
                        res_beta = beta
                        res_gamma = gamma
                        error = mse
                        res = model_fit
        return HW_Prediccion(res, res_alpha, res_beta, res_gamma)

In [94]:
modelo_calibrado_wh = HW_calibrado(train_em, test_em)
modelo_calibrado_wh_fit = modelo_calibrado_wh.fit(0.05)

In [95]:
modelo_calibrado_wh_fit.alpha

np.float64(0.8)

In [96]:
modelo_calibrado_wh_fit.beta

np.float64(0.1)

In [97]:
modelo_calibrado_wh_fit.gamma

np.float64(0.1)

In [98]:
pred_calibrado_wh = modelo_calibrado_wh_fit.forecast(3)
pred_calibrado_wh

2026-01-01    1.490728e+06
2026-02-01    1.464647e+06
2026-03-01    1.504117e+06
Freq: MS, dtype: float64

#### Redes Neuronales

In [99]:
class LSTM_TSPrediccion(Prediccion):
    def __init__(self, modelo):
        super().__init__(modelo)
        self.__scaler = MinMaxScaler(feature_range=(0, 1))
        self.__X = self.__scaler.fit_transform(self.modelo.ts.to_frame())

    def __split_sequence(self, sequence, n_steps):
        X, y = [], []
        for i in range(n_steps, len(sequence)):
            X.append(self.__X[i - n_steps : i, 0])
            y.append(self.__X[i, 0])
        return np.array(X), np.array(y)

    def forecast(self, steps=1):
        res = []
        p = self.modelo.p
        for i in range(steps):
            y_pred = np.array(self.__X[-p:]).reshape(
                1, p, 1
            )  # y_pred = [self.__X[-p:].tolist()]
            X, y = self.__split_sequence(self.__X, p)
            X = np.reshape(X, (X.shape[0], X.shape[1], 1))
            self.modelo.m.fit(X, y, epochs=10, batch_size=1, verbose=0)
            pred = self.modelo.m.predict(y_pred)
            res.append(self.__scaler.inverse_transform(pred).tolist()[0][0])
            self.__X = np.append(self.__X, pred.tolist(), axis=0)

        start = self.modelo.ts.index[-1]
        freq = self.modelo.ts.index.freqstr
        fechas = pd.date_range(start=start, periods=steps + 1, freq=freq)
        fechas = fechas.delete(0)
        res = pd.Series(res, index=fechas)
        return res


class LSTM_TS(Modelo):
    def __init__(
        self, ts, p=1, lstm_units=50, dense_units=1, optimizer="rmsprop", loss="mse"
    ):
        super().__init__(ts)
        self.__p = p
        self.__m = Sequential()
        self.__m.add(LSTM(units=lstm_units, input_shape=(p, 1)))
        self.__m.add(Dense(units=dense_units))
        self.__m.compile(optimizer=optimizer, loss=loss)

    @property
    def m(self):
        return self.__m

    @property
    def p(self):
        return self.__p

    def fit(self):
        res = LSTM_TSPrediccion(self)
        return res

In [100]:
modelo_dl = LSTM_TS(train_em, 3)
modelo_dl_fit = modelo_dl.fit()

In [101]:
pred_dl = modelo_dl_fit.forecast(3)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step


In [102]:
pred_dl

2026-01-01    1676820.250
2026-02-01    1595657.375
2026-03-01    1592434.250
Freq: MS, dtype: float64

#### Gráfico de la predicción

In [110]:
fig = go.Figure()

# train
no_plot = fig.add_trace(
    go.Scatter(
        x=train_em.index.tolist(),
        y=train_em.values.tolist(),
        mode="lines+markers",  # lines+markers
        name="Train",
    )
)

# test real
no_plot = fig.add_trace(
    go.Scatter(
        x=test_em.index.tolist(),
        y=test_em.values.tolist(),
        mode="lines+markers",
        name="Test",
    )
)

# pred Holt-Winters
no_plot = fig.add_trace(
    go.Scatter(
        x=pred_holt_winters.index.tolist(),
        y=pred_holt_winters.values.tolist(),
        mode="lines+markers",
        name="Holt-Winters",
    )
)

# pred Hold-Winters calibrado
no_plot = fig.add_trace(
    go.Scatter(
        x=pred_calibrado_wh.index.tolist(),
        y=pred_calibrado_wh.values.tolist(),
        mode="lines+markers",
        name="Holt-Winters Calibrado",
    )
)

# pred Deep Learning
no_plot = fig.add_trace(
    go.Scatter(
        x=pred_dl.index.tolist(),
        y=pred_dl.values.tolist(),
        mode="lines+markers",
        name="Deep Learning",
    )
)

# # pred drift
# no_plot = fig.add_trace(
#     go.Scatter(x=pred_drift_ts.index.tolist(),y=pred_drift_ts.values.tolist(), mode = 'lines+markers',
#                name="Drift"))

no_plot = fig.update_xaxes(rangeslider_visible=True)
fig.show()
fig.write_html(
    "grafico_interactivo_ejercicio1_prediccion.html",
    include_plotlyjs="cdn"
)

### Parte 4
- d) Con un gráfico mida el error de cada uno de los modelos anteriores y determine cual de los modelos es el mejor.

In [104]:
# pd.DataFrame({
#     'Real': test_em,
#     'HW': pred_holt_winters,
#     'HW_calibrado': pred_calibrado_wh,
#     'LSTM': pred_dl
# })

In [111]:
errores_em = ts_error(
    [pred_holt_winters, pred_calibrado_wh, pred_dl],
    test_em,
    ["Holt-Winters", "Holt-Winters Calibrado", "Deep Learning"],
)
errores_em.df_errores()

,MSE,RMSE,RE,CORR
Holt-Winters,5.939771e+08,24371.646226,0.013338,0.552920
Holt-Winters Calibrado,1.800457e+07,4243.179715,0.001896,0.998520
Deep Learning,1.921467e+10,138617.003911,0.089149,0.095275


In [112]:
fig = errores_em.plotly_errores()
fig.show()
fig.write_html(
    "grafico_interactivo_ejercicio1_errores.html",
    include_plotlyjs="cdn"
)

`Holt-Winters Calibrado` es el modelo que mejor desempeño general al obtener los menores errores (MSE, RMSE y RE).   
Con tan solo 18,004,570 de `MSE`, un `RMSE` mucho menor al resto 4243.18, al igual que el RE con 0.001896 y una `CORR` la mayor correlación respecto a los valores reales, con practicamente un 1.

### Parte 5

- e) Con el mejor modelo encontrado en el punto anterior genere la predicción de 3 meses, pero esta vez utilizando toda la serie de tiempo. Grafique la serie original y la predicción.

In [107]:
modelo_final = ExponentialSmoothing(emision_monetaria_ts, trend="add", seasonal="add")
modelo_final_fit = modelo_final.fit(smoothing_level=0.8, smoothing_slope=0.1, smoothing_seasonal=0.1)
pred_final = modelo_final_fit.forecast(3)
pred_final

2026-04-01    1.508210e+06
2026-05-01    1.494829e+06
2026-06-01    1.507990e+06
Freq: MS, dtype: float64

In [113]:
fig = go.Figure()
no_plot = fig.add_trace(
    go.Scatter(
        x=emision_monetaria_ts.index.tolist(),
        y=emision_monetaria_ts.values.tolist(),
        mode="lines+markers",
        name="Original",
    )
)
no_plot = fig.add_trace(
    go.Scatter(
        x=pred_final.index.tolist(),
        y=pred_final.values.tolist(),
        mode="lines+markers",
        name="Predicción",
    )
)

no_plot = fig.update_xaxes(rangeslider_visible=True)
fig.show()

fig.write_html(
    "grafico_interactivo_ejercicio1_prediccion_final.html",
    include_plotlyjs="cdn"
)

<a id="ej2"></a>
# Ejercicio 2
[35 puntos] La tabla `traficoweb_v1.csv` contiene la cantidad de usuarios que ingresan a una determinada página web por día desde el 14 de setiembre del 2014 al 19 de agosto del 2020.
Con la tabla realice lo siguiente:
- a) Verifique si hay fechas faltantes y de ser así relice la corrección mediante un suavizado (Utilice el valor que usted considere).
- b) Convierta a serie de tiempo.
- c) Usando las últimas 2 semanas para pruebas y el resto de fechas para entrenamiento genere un modelo con `HOLT-WINTERS`, `HOLT-WINTERS Calibrado` y `Redes Neuronales`.
- d) Con un gráfico mida el error de cada uno de los modelos anteriores y determine cuál de los modelos es el mejor.
- e) Con el mejor modelo encontrado en el punto anterior genere la predicción de una semana, pero esta vez utilizando toda la serie de tiempo. Grafique la serie original y la predicción.


[↑ Volver al Menú](#menu)


### Preparación de los datos

In [117]:
traficoweb = pd.read_csv("../../w1/h1/traficoweb_v1.csv", delimiter=";")
traficoweb

,Fecha,Visitas
0,9/14/2014,1582
1,9/15/2014,2528
2,9/16/2014,2630
3,9/17/2014,2614
4,9/18/2014,2366
...,...,...
2155,8/15/2020,1696
2156,8/16/2020,2037
2157,8/17/2020,2638
2158,8/18/2020,2683


In [118]:
traficoweb["Visitas"] = traficoweb["Visitas"].astype("int64")
traficoweb["Fecha"] = traficoweb["Fecha"].astype("datetime64[ns]")

### Parte 1

- a) Verifique si hay fechas faltantes y de ser así relice la corrección mediante un suavizado (Utilice el valor que usted considere).
- b) Convierta a serie de tiempo.

In [119]:
faltan_fechas = hay_fechas_faltantes(traficoweb, col_name_date="Fecha", frequency="D")

Sí, hay -7 'Fecha' faltantes.


In [120]:
len(faltan_fechas["date_range"])

2167

In [121]:
traficoweb["Visitas"].isnull().sum()

np.int64(0)

In [122]:
traficoweb_filled = rellenar_fechas(
    traficoweb,
    col_name_date="Fecha",
    serie_fechas_completas=faltan_fechas["date_range"],
)
print("len: ", len(traficoweb_filled))

Primeras 5 filas:
 [Timestamp('2014-10-01 00:00:00'), Timestamp('2015-04-28 00:00:00'), Timestamp('2018-07-09 00:00:00'), Timestamp('2019-09-21 00:00:00'), Timestamp('2019-12-22 00:00:00')]

Valores faltantes:  0
len:  2167


In [123]:
traficoweb_filled["Visitas"].isnull().sum()

np.int64(7)

Como tiee 7 valores faltantes, por las fechas que hacian falta, se hace un suavizado móvil de 3 datos, pero primero se transforma a serie de tiempo

In [124]:
tw_filled_ts = to_serie_tiempo(traficoweb_filled, "Fecha", "Visitas", frequency="D")

In [125]:
tw_filled_ts_3 = imputar_valores_faltantes_suavizados(
    traficoweb_filled,
    col_name_values="Visitas",
    size=3,
    to_serie=True,
    frequency="D",
    col_name_date="Fecha",
)

Todos los valores faltantes fueron imputados con suavizado de 3 datos.


### Parte 2
- c) Usando las últimas 2 semanas para pruebas y el resto de fechas para entrenamiento genere un modelo con `HOLT-WINTERS`, `HOLT-WINTERS Calibrado` y `Redes Neuronales`.

In [126]:
day_test = 14
tw_filled_ts_3_train_test = train_test(tw_filled_ts_3, day_test)

Train: 2153 
Test : 14


In [127]:
def modelos_ts(ts, test_size, forecast, periods=None, is_holt_winters = False, is_holt_winters_calibrado = False, is_lstm = False):
    train_test_division = train_test(ts, test_size)
    if is_holt_winters is True:
        holt_winters = ExponentialSmoothing(train_test_division['train'], trend = 'add', seasonal = 'add', seasonal_periods=periods)
        holt_winters_fit = holt_winters.fit()
        pred = holt_winters_fit.forecast(forecast)
    
    elif is_holt_winters_calibrado is True:
        hw_calibrado = HW_calibrado(train_test_division['train'], train_test_division['test'], seasonal_periods=periods)
        hw_calibrado_fit = hw_calibrado.fit(0.05)
        pred = hw_calibrado_fit.forecast(forecast)
        print(
            'alpha: ', hw_calibrado_fit.alpha,
            '\nbeta : ', hw_calibrado_fit.beta,
            '\ngamma: ', hw_calibrado_fit.gamma)
        
    elif is_lstm is True:
        lstm = LSTM_TS(train_test_division['train'])
        lstm_fit = lstm.fit()
        pred = lstm_fit.forecast(forecast)
    
    return pred

In [128]:
pred_holt_winters = modelos_ts(tw_filled_ts_3, test_size=day_test, periods=7, forecast=day_test, is_holt_winters=True)

Train: 2153 
Test : 14


#### Holt-Winters

In [129]:
pred_holt_winters[:5]

2020-08-06    2591.443837
2020-08-07    1898.812124
2020-08-08    1021.367966
2020-08-09    1544.124393
2020-08-10    2663.600569
Freq: D, dtype: float64

#### Holt-Winters Calibrado

In [130]:
pred_hw_calibrado = modelos_ts(tw_filled_ts_3, test_size=day_test, periods=7, forecast=day_test, is_holt_winters_calibrado=True)

Train: 2153 
Test : 14
alpha:  0.6000000000000001 
beta :  0.05 
gamma:  0.35000000000000003


#### Redes Neuronales

In [131]:
pred_lstm = modelos_ts(tw_filled_ts_3, test_size=day_test, forecast=day_test, is_lstm=True)

Train: 2153 
Test : 14
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step


In [132]:
def predictions_chart(train, test, pred_holt_winters, pred_calibrado_wh, pred_dl, lines="lines+markers", namechart=""):
    fig = go.Figure()

    # train
    no_plot = fig.add_trace(
        go.Scatter(
            x=train.index.tolist(),
            y=train.values.tolist(),
            mode=lines,  # lines+markers
            name="Train",
        )
    )

    # test real
    no_plot = fig.add_trace(
        go.Scatter(
            x=test.index.tolist(),
            y=test.values.tolist(),
            mode=lines,
            name="Test",
        )
    )

    # pred Holt-Winters
    no_plot = fig.add_trace(
        go.Scatter(
            x=pred_holt_winters.index.tolist(),
            y=pred_holt_winters.values.tolist(),
            mode=lines,
            name="Holt-Winters",
        )
    )

    # pred Hold-Winters calibrado
    no_plot = fig.add_trace(
        go.Scatter(
            x=pred_calibrado_wh.index.tolist(),
            y=pred_calibrado_wh.values.tolist(),
            mode=lines,
            name="Holt-Winters Calibrado",
        )
    )

    # pred Deep Learning
    no_plot = fig.add_trace(
        go.Scatter(
            x=pred_dl.index.tolist(),
            y=pred_dl.values.tolist(),
            mode=lines,
            name="Deep Learning",
        )
    )

    no_plot = fig.update_xaxes(rangeslider_visible=True)
    fig.show()
    fig.write_html(
        f"grafico_interactivo_{namechart}.html",
        include_plotlyjs="cdn"
    )

#### Gráfico de predicción

In [133]:
predictions_chart(tw_filled_ts_3_train_test['train'],
                 tw_filled_ts_3_train_test['test'],
                 pred_holt_winters,
                 pred_hw_calibrado,
                 pred_lstm,
                 lines="lines",
                 namechart="ejercicio2_prediccion")

### Parte 3
- d) Con un gráfico mida el error de cada uno de los modelos anteriores y determine cuál de los modelos es el mejor.

In [134]:
errores_tw = ts_error(
    [pred_holt_winters, pred_hw_calibrado, pred_lstm],
    tw_filled_ts_3_train_test['test'],
    ["Holt-Winters", "Holt-Winters Calibrado", "Deep Learning"],
)
errores_tw.df_errores()

,MSE,RMSE,RE,CORR
Holt-Winters,222983.090528,472.210854,0.149007,0.727784
Holt-Winters Calibrado,85241.903653,291.962161,0.066027,0.772487
Deep Learning,407036.545069,637.994158,0.199254,0.179224


In [169]:
# errores_em.plot_errores()
fig = errores_tw.plotly_errores()
fig.show()
fig.write_html(
    "grafico_interactivo_ejercicio2_errores.html",
    include_plotlyjs="cdn"
)

En términos generales, `Holt-Winters Calibrado` presenta el mejor desempeño general al obtener los menores errores (MSE, RMSE y RE):
- Menor `MSE` con 85241.90
- Menor `RMSE` con 291.96
- Un `RE` mucho menor al resto 0.066027
- Y un `CORR`, la mayor correlación respecto a los valores reales bueno, y más alto que los otros dos modelos, con 0.77

### Parte 4
- e) Con el mejor modelo encontrado en el punto anterior genere la predicción de una semana, pero esta vez utilizando toda la serie de tiempo. Grafique la serie original y la predicción.

In [136]:
modelo_hwc_final = ExponentialSmoothing(tw_filled_ts_3, trend="add", seasonal="add")
modelo_hwc_final_fit = modelo_hwc_final.fit(smoothing_level=0.60, smoothing_slope=0.05, smoothing_seasonal=0.35)
pred_hwc_final = modelo_hwc_final_fit.forecast(7)
pred_hwc_final

2020-08-20    1828.619399
2020-08-21    1420.907327
2020-08-22     742.760523
2020-08-23    1037.086623
2020-08-24    1660.583134
2020-08-25    1702.525398
2020-08-26    1706.487079
Freq: D, dtype: float64

In [171]:
fig = go.Figure()
no_plot = fig.add_trace(
    go.Scatter(
        x=tw_filled_ts_3.index.tolist(),
        y=tw_filled_ts_3.values.tolist(),
        mode="lines+markers",
        name="Original",
    )
)
no_plot = fig.add_trace(
    go.Scatter(
        x=pred_hwc_final.index.tolist(),
        y=pred_hwc_final.values.tolist(),
        mode="lines+markers",
        name="Predicción",
    )
)

no_plot = fig.update_xaxes(rangeslider_visible=True)
fig.show()
fig.write_html(
    "grafico_interactivo_ejercicio2_prediccion_final.html",
    include_plotlyjs="cdn"
)

<a id="ej3"></a>
# Ejercicio 3 
[35 puntos] La tabla `consumo_agua.csv` contiene el consumo de agua diario de una región desde el 02 de marzo del 2019 hasta el 30 de noviembre de 2021. Con la tabla realice lo siguiente:
- a) Convierta a serie de tiempo.
- b) Usando el último mes para pruebas y el resto de fechas para entrenamiento genere los modelos de `HOLT-WINTERS`, `HOLT-WINTERS Calibrado` y `Redes Neuronales`, luego en un solo gráfico muestre la serie de entrenamiento, la serie de prueba y el resultado de la predicción de cada uno de los modelos anteriores.
- c) Con un gráfico mida el error y determine cuál modelo es el mejor.
- d) Con el mejor modelo encontrado en el punto anterior genere la predicción de un mes, pero esta vez utilizando toda la serie de tiempo.
- e) Según el comportamiento de años anteriores el día 26 de diciembre se consume menos agua de lo normal. Por tanto, genere una regla para dicha fecha y luego grafique la serie original junto con la predicción.

**NOTA: Toda conversión a serie de tiempo debe incluir la frecuencia correspondiente.**

[↑ Volver al Menú](#menu)

### Parte 1
- a) Convierta a serie de tiempo.

In [139]:
consumo_agua = pd.read_csv("consumo_agua.csv", delimiter=";")
consumo_agua

,fecha,valor
0,2019-03-02,1771.416
1,2019-03-03,3163.136
2,2019-03-04,3139.684
3,2019-03-05,3170.836
4,2019-03-06,3062.992
...,...,...
1000,2021-11-26,4528.632
1001,2021-11-27,4635.904
1002,2021-11-28,4658.300
1003,2021-11-29,4568.320


In [140]:
consumo_agua.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   fecha   1005 non-null   object 
 1   valor   1005 non-null   float64
dtypes: float64(1), object(1)
memory usage: 15.8+ KB


In [141]:
consumo_agua["fecha"] = consumo_agua["fecha"].astype("datetime64[ns]")
consumo_agua.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   fecha   1005 non-null   datetime64[ns]
 1   valor   1005 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 15.8 KB


In [142]:
consumo_agua["fecha"].isnull().sum()

np.int64(0)

In [143]:
# recordar que esta funcion devuelve un dic con 'date_range' completo y 'faltantes' que son las fechas faltantes, si las hay.
faltan_fechas = hay_fechas_faltantes(consumo_agua, col_name_date="fecha", frequency="D")

No, no hay valores faltantes en la columna 'fecha'.


In [144]:
consumo_agua["valor"].isnull().sum()

np.int64(0)

In [145]:
# pasamos a series de tiempo usando la funcion del ejercicio 2
# usamos los datos rellenados para no perder la continuidad de la serie de tiempo, los nombres de las cols y la frecuencia.
consumo_agua_ts = to_serie_tiempo(consumo_agua, col_name_date="fecha", col_name_values="valor", frequency="D")
consumo_agua_ts[:10]

fecha
2019-03-02    1771.416
2019-03-03    3163.136
2019-03-04    3139.684
2019-03-05    3170.836
2019-03-06    3062.992
2019-03-07    3027.308
2019-03-08    2937.108
2019-03-09    2920.432
2019-03-10    2636.060
2019-03-11    1127.608
Freq: D, Name: valor, dtype: float64

### Parte 2
- b) Usando el último mes para pruebas y el resto de fechas para entrenamiento genere los modelos de `HOLT-WINTERS`, `HOLT-WINTERS Calibrado` y `Redes Neuronales`, luego en un solo gráfico muestre la serie de entrenamiento, la serie de prueba y el resultado de la predicción de cada uno de los modelos anteriores.

In [146]:
days_test = 30 # días => 1 mes
train_test_consumo_agua = train_test(consumo_agua_ts, test_size=days_test)

Train: 975 
Test : 30


#### Holt-Winters

In [147]:
consumoAgua_ts_hw = modelos_ts(consumo_agua_ts, test_size=days_test, forecast=days_test, is_holt_winters=True)

Train: 975 
Test : 30


#### Holt-Winters Calibrado

In [148]:
consumoAgua_ts_hw_calibrado = modelos_ts(consumo_agua_ts, test_size=days_test, forecast=days_test, is_holt_winters_calibrado=True)

Train: 975 
Test : 30
alpha:  0.2 
beta :  0.0 
gamma:  0.7000000000000001


#### Redes Neuronales

In [149]:
consumoAgua_ts_lstm = modelos_ts(consumo_agua_ts, test_size=days_test, forecast=days_test, is_lstm=True)

Train: 975 
Test : 30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s

#### Gráfico de Predicción

In [172]:
predictions_chart(train_test_consumo_agua['train'],
                 train_test_consumo_agua['test'],
                 consumoAgua_ts_hw,
                 consumoAgua_ts_hw_calibrado,
                 consumoAgua_ts_lstm,
                 lines="lines", 
                 namechart="ejercicio3_prediccion")

### Parte 3

- c) Con un gráfico mida el error y determine cuál modelo es el mejor.

In [151]:
errores_ca = ts_error(
    [consumoAgua_ts_hw, consumoAgua_ts_hw_calibrado, consumoAgua_ts_lstm],
    train_test_consumo_agua['test'],
    ["Holt-Winters", "Holt-Winters Calibrado", "Redes Neuronales (LSTM)"],
)
errores_ca.df_errores()

,MSE,RMSE,RE,CORR
Holt-Winters,2.574231e+05,507.368775,0.075541,0.790842
Holt-Winters Calibrado,2.098204e+05,458.061568,0.064934,0.802965
Redes Neuronales (LSTM),1.335764e+06,1155.752459,0.264965,0.088280


In [173]:
fig = errores_ca.plotly_errores()
fig.show()
fig.write_html(
    "grafico_interactivo_ejercicio3_errores.html",
    include_plotlyjs="cdn"
)

Se puede apreciar en el gráfico así como en la tabla anterior, que `Holt-Winters` y `Holt-Winters Calibrado` presentan un rendimiento similar, sin embargo en términos generales `Holt-Winters Calibrado` sería el mejor de estos, aunque la mejora es moderada, porque tiene:
- Menor `MSE` con 209820.4
- Menor `RMSE` con 458.06
- Un `RE` mucho menor al resto 0.064934
- Y un `CORR`, la mayor correlación respecto a los valores reales con 0.802965

### Parte 4
- d) Con el mejor modelo encontrado en el punto anterior genere la predicción de un mes, pero esta vez utilizando toda la serie de tiempo.

In [153]:
model_hwc_final = ExponentialSmoothing(consumo_agua_ts, trend="add", seasonal="add")
model_hwc_final_fit = model_hwc_final.fit(smoothing_level=0.20, smoothing_slope=0.00, smoothing_seasonal=0.70)

pred_hwc_final_ca = model_hwc_final_fit.forecast(31)
pred_hwc_final_ca[:10]

2021-12-01    4374.724565
2021-12-02    5032.344472
2021-12-03    4940.640661
2021-12-04    4903.547137
2021-12-05    5012.693008
2021-12-06    4932.388683
2021-12-07    3119.928811
2021-12-08    4388.467311
2021-12-09    5046.087217
2021-12-10    4954.383406
Freq: D, dtype: float64

In [174]:
fig = go.Figure()
no_plot = fig.add_trace(
    go.Scatter(
        x=consumo_agua_ts.index.tolist(),
        y=consumo_agua_ts.values.tolist(),
        mode="lines+markers",
        name="Original",
    )
)
no_plot = fig.add_trace(
    go.Scatter(
        x=pred_hwc_final_ca.index.tolist(),
        y=pred_hwc_final_ca.values.tolist(),
        mode="lines+markers",
        name="Predicción",
    )
)

no_plot = fig.update_xaxes(rangeslider_visible=True)
fig.show()
fig.write_html(
    "grafico_interactivo_ejercicio3_prediccion_final.html",
    include_plotlyjs="cdn"
)

### Parte 5
- e) Según el comportamiento de años anteriores el día 26 de diciembre se consume menos agua de lo normal. Por tanto, genere una regla para dicha fecha y luego grafique la serie original junto con la predicción.

Cálculo del Factor de Ajuste en la Regla

In [155]:
regla_26_dec = consumo_agua_ts[consumo_agua_ts.index < "2020-12-26"]
regla_26_dec.tail()

fecha
2020-12-21    4217.684
2020-12-22    4391.572
2020-12-23    4480.804
2020-12-24    4392.496
2020-12-25    4052.068
Freq: D, Name: valor, dtype: float64

Se genera el "mejor modelo", según la fase anterior

In [156]:
modelo_regla_hw_cal = ExponentialSmoothing(regla_26_dec, trend = 'add', seasonal = 'add')
modelo_regla_hw_cal_fit = modelo_regla_hw_cal.fit(smoothing_level=0.20, smoothing_slope=0.00, smoothing_seasonal=0.70)

Predecimos solo la fecha, es decir, el 26 de Dic

In [157]:
pred_26_dec = modelo_regla_hw_cal_fit.forecast(1)
pred_26_dec

2020-12-26    2759.74869
Freq: D, dtype: float64

Tomamos el valor real de la serie completa

In [158]:
real_26_dec = consumo_agua_ts[consumo_agua_ts.index == "2020-12-26"]
real_26_dec

fecha
2020-12-26    1797.42
Freq: D, Name: valor, dtype: float64

In [159]:
error = pred_26_dec.values[0] - real_26_dec.values[0]
error

np.float64(962.3286903307544)

El valor del error fue positivo 962.33

Calculando el Factor de Ajuste

In [160]:
# Observar que debemos sumar 1 si el error es por debajo del real y restar 1 si el valor esta por encima del real.
if error < 0:
  factor_ajuste = 1 + (abs(error) / pred_26_dec.values[0])
else:
  factor_ajuste = 1 - (abs(error) / pred_26_dec.values[0])

factor_ajuste

np.float64(0.6512984339108718)

Verficamos que la regla sea correcta

In [161]:
pred_26_dec.values[0] * factor_ajuste

np.float64(1797.42)

Generamos el modelo y la predicción, pero con toda la serie de tiempo.

In [162]:
model_hwc_final = ExponentialSmoothing(consumo_agua_ts, trend="add", seasonal="add")
model_hwc_final_fit = model_hwc_final.fit(smoothing_level=0.20, smoothing_slope=0.00, smoothing_seasonal=0.70)
pred = model_hwc_final_fit.forecast(31)
pred[20:28]

2021-12-21    3147.414301
2021-12-22    4415.952801
2021-12-23    5073.572707
2021-12-24    4981.868896
2021-12-25    4944.775373
2021-12-26    5053.921243
2021-12-27    4973.616918
2021-12-28    3161.157046
Freq: D, dtype: float64

Aplicamos la regla solo al día 26 de Dic

In [163]:
pred[pred.index == "2021-12-26"] = pred[pred.index == "2021-12-26"] * factor_ajuste
pred[pred.index == "2021-12-26"]

2021-12-26    3291.610991
Freq: D, dtype: float64

In [164]:
# Como un agregado a estas reglas es conveniente utilizar el máximo o el mínimo si se pasa la predicción de alguno de estos 2 rangos.
if pred[pred.index == "2021-12-26"].values > max(pred): 
  pred[pred.index == "2021-12-26"] = max(pred)

if pred[pred.index == "2021-12-26"].values < min(pred): 
  pred[pred.index == "2021-12-26"] = min(pred)

#### Gráfico de predicción

In [176]:
fig = go.Figure()
no_plot = fig.add_trace(
  go.Scatter(x = consumo_agua_ts.index.tolist(), y = consumo_agua_ts.values.tolist(), 
             mode = 'lines+markers', name = "Original")
)
no_plot = fig.add_trace(
  go.Scatter(x = pred.index.tolist(), y = pred.values.tolist(), 
             mode = 'lines+markers', name = "Predicción")
)

# no_plot = fig.add_annotation(x = "2021-12-26", y = pred[15], text = "26 de Dic")

no_plot = fig.update_xaxes(rangeslider_visible=True)
fig.show()
fig.write_html(
    "grafico_interactivo_ejercicio3_prediccion_regla.html",
    include_plotlyjs="cdn"
)